# Brain-to-Text RNN Training and Evaluation

This notebook implements training and evaluation for the Brain-to-Text RNN model.

## Setup Instructions

1. Upload the following files to your Google Drive in a folder (e.g., `MyDrive/brain-to-text/`):
   - `model_training/train_model.py`
   - `model_training/rnn_trainer.py`
   - `model_training/rnn_model.py`
   - `model_training/dataset.py`
   - `model_training/data_augmentations.py`
   - `model_training/evaluate_model.py`
   - `model_training/evaluate_model_helpers.py`
   - `model_training/rnn_args.yaml` (or `rnn_diphone_args.yaml`)
   - `data/t15_copyTaskData_description.csv`
   - Your HDF5 data files (in the structure expected by your config)

2. Update the `DRIVE_PATH` variable below to point to your folder


## 1. Install Dependencies


In [ ]:
# Install required packages
%pip install torch torchaudio omegaconf h5py numpy pandas tqdm scipy editdistance -q


## 2. Mount Google Drive and Set Paths


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# SET THIS PATH to your folder containing the uploaded files
# Example: '/content/drive/MyDrive/brain-to-text'
DRIVE_PATH = '/content/drive/MyDrive/brain-to-text'  # UPDATE THIS!

# Verify the path exists
if not os.path.exists(DRIVE_PATH):
    print(f"ERROR: Path {DRIVE_PATH} does not exist!")
    print("Please update DRIVE_PATH to point to your folder.")
else:
    print(f"Using path: {DRIVE_PATH}")
    print(f"Contents: {os.listdir(DRIVE_PATH)}")


## 3. Set Up Working Directory


In [ ]:
import sys
import shutil

# Create a working directory
WORK_DIR = '/content/brain_to_text_workspace'
os.makedirs(WORK_DIR, exist_ok=True)

# Copy Python files from Drive to working directory
python_files = [
    'train_model.py',
    'rnn_trainer.py',
    'rnn_model.py',
    'dataset.py',
    'data_augmentations.py',
    'evaluate_model.py',
    'evaluate_model_helpers.py',
]

for file in python_files:
    src = os.path.join(DRIVE_PATH, file)
    dst = os.path.join(WORK_DIR, file)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {file}")
    else:
        print(f"WARNING: {file} not found at {src}")

# Copy YAML config file
config_files = ['rnn_args.yaml', 'rnn_diphone_args.yaml']
for config_file in config_files:
    src = os.path.join(DRIVE_PATH, config_file)
    if os.path.exists(src):
        dst = os.path.join(WORK_DIR, config_file)
        shutil.copy(src, dst)
        print(f"Copied {config_file}")

# Change to working directory
os.chdir(WORK_DIR)
print(f"\nWorking directory: {os.getcwd()}")

# Add to Python path
sys.path.insert(0, WORK_DIR)


## 4. Update Config File Paths

Update the paths in your YAML config file to point to your Google Drive data location.


In [ ]:
from omegaconf import OmegaConf

# Path to your config file (update if using a different one)
CONFIG_FILE = 'rnn_diphone_args.yaml'  # or 'rnn_args.yaml'

# Path to your data directory on Google Drive
# Example: '/content/drive/MyDrive/brain-to-text/data/hdf5_data_final'
DATA_DIR_ON_DRIVE = '/content/drive/MyDrive/brain-to-text/data/hdf5_data_final'  # UPDATE THIS!

# Path to CSV file on Google Drive
CSV_PATH_ON_DRIVE = '/content/drive/MyDrive/brain-to-text/data/t15_copyTaskData_description.csv'  # UPDATE THIS!

# Load config
if os.path.exists(CONFIG_FILE):
    args = OmegaConf.load(CONFIG_FILE)
    
    # Update dataset directory path
    args.dataset.dataset_dir = DATA_DIR_ON_DRIVE
    
    # Update output directory to be in Drive (so checkpoints persist)
    args.output_dir = os.path.join(DRIVE_PATH, 'trained_models', args.output_dir.split('/')[-1])
    args.checkpoint_dir = os.path.join(args.output_dir, 'checkpoint')
    
    # Set GPU number to 0 (Colab typically has one GPU)
    args.gpu_number = '0'
    
    # Save updated config
    OmegaConf.save(args, CONFIG_FILE)
    print(f"Updated config saved to {CONFIG_FILE}")
    print(f"Dataset dir: {args.dataset.dataset_dir}")
    print(f"Output dir: {args.output_dir}")
else:
    print(f"ERROR: Config file {CONFIG_FILE} not found!")


## 5. Training

Run this cell to start training the model.


In [ ]:
from omegaconf import OmegaConf
from rnn_trainer import BrainToTextDecoder_Trainer

# Load config
args = OmegaConf.load(CONFIG_FILE)

# Initialize trainer
trainer = BrainToTextDecoder_Trainer(args)

# Train model
print("Starting training...")
metrics = trainer.train()

print("\nTraining completed!")
print(f"Best validation PER: {trainer.best_val_PER:.4f}")


## 6. Evaluation

Run this cell to evaluate a trained model. Update the `MODEL_PATH` variable to point to your trained model directory.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from omegaconf import OmegaConf
import time
from tqdm import tqdm
import editdistance

from rnn_model import GRUDecoder
from evaluate_model_helpers import *

# ========== CONFIGURATION ==========
# Path to trained model directory (update this!)
# Example: '/content/drive/MyDrive/brain-to-text/trained_models/diphone_rnn'
MODEL_PATH = '/content/drive/MyDrive/brain-to-text/trained_models/diphone_rnn'  # UPDATE THIS!

# Path to data directory
DATA_DIR = DATA_DIR_ON_DRIVE

# Path to CSV file
CSV_PATH = CSV_PATH_ON_DRIVE

# Evaluation type: 'val' for validation set, 'test' for test set
EVAL_TYPE = 'val'  # or 'test'

# GPU number (0 for Colab, -1 for CPU)
GPU_NUMBER = 0
# ===================================

# Load CSV file
b2txt_csv_df = pd.read_csv(CSV_PATH)

# Load model args
model_args = OmegaConf.load(os.path.join(MODEL_PATH, 'checkpoint/args.yaml'))

# Determine if model was trained with diphones
use_diphones = model_args.get('use_diphones', False)

# Set up GPU device
if torch.cuda.is_available() and GPU_NUMBER >= 0:
    if GPU_NUMBER >= torch.cuda.device_count():
        raise ValueError(f'GPU number {GPU_NUMBER} is out of range. Available GPUs: {torch.cuda.device_count()}')
    device = torch.device(f'cuda:{GPU_NUMBER}')
    print(f'Using {device} for model inference.')
else:
    if GPU_NUMBER >= 0:
        print(f'GPU number {GPU_NUMBER} requested but not available.')
    print('Using CPU for model inference.')
    device = torch.device('cpu')

# Define model
model = GRUDecoder(
    neural_dim = model_args['model']['n_input_features'],
    n_units = model_args['model']['n_units'], 
    n_days = len(model_args['dataset']['sessions']),
    n_classes = model_args['dataset']['n_classes'],
    rnn_dropout = model_args['model']['rnn_dropout'],
    input_dropout = model_args['model']['input_network']['input_layer_dropout'],
    n_layers = model_args['model']['n_layers'],
    patch_size = model_args['model']['patch_size'],
    patch_stride = model_args['model']['patch_stride'],
)

# Load model weights
checkpoint_path = os.path.join(MODEL_PATH, 'checkpoint/best_checkpoint')
if torch.device(device).type == 'cuda':
    checkpoint = torch.load(checkpoint_path, weights_only=False, map_location=device)
else:
    checkpoint = torch.load(checkpoint_path, weights_only=False, map_location=torch.device('cpu'))

# Rename keys to not start with "module." (happens if model was saved with DataParallel)
for key in list(checkpoint['model_state_dict'].keys()):
    if key.startswith('module.'):
        new_key = key.replace('module.', '')
        checkpoint['model_state_dict'][new_key] = checkpoint['model_state_dict'].pop(key)
    if key.startswith('_orig_mod.'):
        new_key = key.replace('_orig_mod.', '')
        checkpoint['model_state_dict'][new_key] = checkpoint['model_state_dict'].pop(key)

model.load_state_dict(checkpoint['model_state_dict'])  

# Add model to device
model.to(device) 

# Set model to eval mode
model.eval()

print(f"Model loaded from {MODEL_PATH}")
print(f"Using diphones: {use_diphones}")


In [ ]:
# Load data for each session
test_data = {}
total_test_trials = 0
for session in model_args['dataset']['sessions']:
    session_dir = os.path.join(DATA_DIR, session)
    if os.path.exists(session_dir):
        files = [f for f in os.listdir(session_dir) if f.endswith('.hdf5')]
        if f'data_{EVAL_TYPE}.hdf5' in files:
            eval_file = os.path.join(session_dir, f'data_{EVAL_TYPE}.hdf5')
            data = load_h5py_file(eval_file, b2txt_csv_df)
            test_data[session] = data
            total_test_trials += len(test_data[session]["neural_features"])
            print(f'Loaded {len(test_data[session]["neural_features"])} {EVAL_TYPE} trials for session {session}.')
    else:
        print(f'Session directory not found: {session_dir}')

print(f'Total number of {EVAL_TYPE} trials: {total_test_trials}')


In [ ]:
# Put neural data through the pretrained model to get phoneme predictions (logits)
with tqdm(total=total_test_trials, desc='Predicting phoneme sequences', unit='trial') as pbar:
    for session, data in test_data.items():
        data['logits'] = []
        data['pred_seq'] = []
        input_layer = model_args['dataset']['sessions'].index(session)
        
        for trial in range(len(data['neural_features'])):
            # Get neural input for the trial
            neural_input = data['neural_features'][trial]

            # Add batch dimension
            neural_input = np.expand_dims(neural_input, axis=0)

            # Convert to torch tensor
            neural_input = torch.tensor(neural_input, device=device, dtype=torch.bfloat16)

            # Run decoding step
            logits = runSingleDecodingStep(neural_input, input_layer, model, model_args, device)
            data['logits'].append(logits)

            pbar.update(1)


In [ ]:
# Convert logits to phoneme sequences and compute metrics
aggregate_edit_distance = 0
total_num_phonemes = 0

for session, data in test_data.items():
    data['pred_seq'] = []
    for trial in range(len(data['logits'])):
        logits = data['logits'][trial][0]
        pred_seq = np.argmax(logits, axis=-1)
        # Remove blanks (0)
        pred_seq = [int(p) for p in pred_seq if p != 0]
        # Remove consecutive duplicates
        pred_seq = [pred_seq[i] for i in range(len(pred_seq)) if i == 0 or pred_seq[i] != pred_seq[i-1]]
        
        # Get trial info
        block_num = data['block_num'][trial]
        trial_num = data['trial_num'][trial]
        
        # Store raw indices for edit distance calculation
        pred_seq_indices = pred_seq.copy()
        
        # Convert indices to human-readable strings for storage and printing
        if use_diphones:
            pred_seq_display = [LOGIT_TO_DIPHONE[p] for p in pred_seq_indices]
        else:
            pred_seq_display = [LOGIT_TO_PHONEME[p] for p in pred_seq_indices]
        
        # Add to data
        data['pred_seq'].append(pred_seq_display)
        
        print(f'Session: {session}, Block: {block_num}, Trial: {trial_num}')
        
        # Only compute metrics if ground truth is available (val split)
        if EVAL_TYPE == 'val':
            true_seq = data['seq_class_ids'][trial][0:data['seq_len'][trial]]
            
            # Convert true sequence indices based on model training
            if use_diphones:
                # Convert phoneme indices to diphone indices (same as trainer does)
                diphone_label = []
                for i in range(len(true_seq) - 1):
                    phone_a = true_seq[i]
                    phone_b = true_seq[i+1]
                    diphone = (LOGIT_TO_PHONEME[phone_a], LOGIT_TO_PHONEME[phone_b])
                    diphone_idx = LOGIT_TO_DIPHONE.index(diphone)
                    diphone_label.append(diphone_idx)
                true_seq_indices = diphone_label
            else:
                true_seq_indices = true_seq
            
            # Convert true sequence to human-readable strings
            if use_diphones:
                true_seq_display = [LOGIT_TO_DIPHONE[p] for p in true_seq_indices]
            else:
                true_seq_display = [LOGIT_TO_PHONEME[p] for p in true_seq_indices]
            
            sentence_label = data['sentence_label'][trial]

            print(f'Sentence label:      {sentence_label}')
            print(f'True sequence:       {" ".join(str(s) for s in true_seq_display)}')
            print(f'Word edit distance:  {editdistance.eval(true_seq_indices, pred_seq_indices)}')
            total_num_phonemes += len(true_seq_indices)
            aggregate_edit_distance += editdistance.eval(true_seq_indices, pred_seq_indices)
        
        print(f'Predicted Sequence:  {" ".join(str(s) for s in pred_seq_display)}')
        print()

unit_name = 'diphones' if use_diphones else 'phonemes'
print(f'Total number of {unit_name}: {total_num_phonemes}')
print(f'Aggregate edit distance: {aggregate_edit_distance}')
if total_num_phonemes > 0:
    print(f'Aggregate PER: {100 * aggregate_edit_distance / total_num_phonemes:.2f}%')


In [ ]:
# Save predicted sequences to CSV file
import pandas as pd

lm_results = {
    'session': [],
    'block': [],
    'trial': [],
    'true_sentence': [],
    'pred_sequence': [],
}

for session in test_data.keys():
    for trial in range(len(test_data[session]['pred_seq'])):
        lm_results['session'].append(session)
        lm_results['block'].append(test_data[session]['block_num'][trial])
        lm_results['trial'].append(test_data[session]['trial_num'][trial])
        if EVAL_TYPE == 'val':
            lm_results['true_sentence'].append(test_data[session]['sentence_label'][trial])
        else:
            lm_results['true_sentence'].append(None)
        # Convert predicted sequence to string
        pred_seq_str = ' '.join(str(s) for s in test_data[session]['pred_seq'][trial])
        lm_results['pred_sequence'].append(pred_seq_str)

# Write to CSV
output_file = os.path.join(MODEL_PATH, f'baseline_rnn_{EVAL_TYPE}_predicted_sequences_{time.strftime("%Y%m%d_%H%M%S")}.csv')
ids = [i for i in range(len(lm_results['pred_sequence']))]
df_out = pd.DataFrame({
    'id': ids, 
    'session': lm_results['session'],
    'block': lm_results['block'],
    'trial': lm_results['trial'],
    'true_sentence': lm_results['true_sentence'],
    'pred_sequence': lm_results['pred_sequence']
})
df_out.to_csv(output_file, index=False)
print(f'Results saved to: {output_file}')
